# IS 477 Final Project

This code was all run on Python 3.12.0.

In [1]:
# imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

The first thing we have to do is read in our datasets. The data has also been provided in a folder and has not been tampered with in any way since it was downloaded. All data manipulation done to it is through this notebook, and running it will provide you with my exact results. However, each dataset has also been provided with a link to the source, in case you want to learn more.

## Dataset 1 - Home Runs

To download this dataset in the exact form I am using here, visit [this website](https://baseballsavant.mlb.com/statcast_search?hfPT=&hfAB=home%5C.%5C.run%7C&hfGT=R%7C&hfPR=&hfZ=&hfStadium=17%7C&hfBBL=&hfNewZones=&hfPull=&hfC=&hfSea=2025%7C&hfSit=&player_type=batter&hfOuts=&home_road=&pitcher_throws=&batter_stands=&hfSA=&hfEventOuts=&hfEventRuns=&game_date_gt=&game_date_lt=&hfMo=&hfTeam=&hfOpponent=&hfRO=&position=&hfInfield=&hfOutfield=&hfInn=&hfBBT=&hfFlag=&metric_1=&group_by=name-event&min_pitches=0&min_results=0&min_pas=0&sort_col=pitches&player_event_sort=api_h_launch_speed&sort_order=desc&chk_event_release_speed=on&chk_event_release_spin_rate=on&chk_event_pitch_name=on&chk_event_launch_speed=on&chk_event_launch_angle=on&chk_event_hit_distance_sc=on#results) and click the rightmost icon in the "Search Results" table header. I renamed my file to `home_runs.csv`. 

In [44]:
hrs = pd.read_csv('data/home_runs.csv')

In [45]:
hrs.head()

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,FF,2025-06-01,92.4,-1.23,6.13,"De La Cruz, Elly",682829,592791,home_run,hit_into_play,...,1.0,1.03,0.43,-0.43,46.3,15.341274,-15.674166,34.381712,32.436840,48.939784
1,CH,2025-06-21,88.3,-2.65,5.13,"Busch, Michael",683737,676106,home_run,hit_into_play,...,1.0,2.74,1.26,-1.26,10.6,24.994119,-14.868757,39.923745,32.076513,40.270063
2,FC,2025-06-17,85.5,2.29,5.64,"Crow-Armstrong, Pete",691718,642239,home_run,hit_into_play,...,2.0,2.96,-0.57,-0.57,38.1,21.267571,-11.614130,24.151153,34.140317,40.713354
3,FF,2025-06-22,95.7,-1.10,5.73,"Suzuki, Seiya",673548,669302,home_run,hit_into_play,...,1.0,1.32,0.91,0.91,32.9,12.957638,-11.024440,25.135264,38.304893,41.312109
4,FF,2025-05-17,92.8,2.30,5.55,"Elko, Tim",671284,571510,home_run,hit_into_play,...,1.0,1.58,1.25,-1.25,26.6,7.762761,10.124528,35.335141,46.279433,27.306206


Now I will filter out only columns that matter for our analysis. These include information about each home run, which player hit it, that player's respective stats, as well as stats on the pitcher throwing the ball and the pitch itself.

In [46]:
hr_cols = ["game_date", "player_name", "launch_speed", "launch_angle", "hit_distance_sc", "pitch_type"]
hrs = hrs[hr_cols]
hrs.head()

,game_date,player_name,launch_speed,launch_angle,hit_distance_sc,pitch_type
0,2025-06-01,"De La Cruz, Elly",117.4,28,423,FF
1,2025-06-21,"Busch, Michael",111.9,26,375,CH
2,2025-06-17,"Crow-Armstrong, Pete",111.5,26,452,FC
3,2025-06-22,"Suzuki, Seiya",111.2,19,389,FF
4,2025-05-17,"Elko, Tim",111.1,22,425,FF


### Data Dictionary

* **`game_date`**

    * This is the date that the home run was hit. We can merge our weather dataset to this one using this key.

* **`player_name`**

    * This is the name of the player that hit the home run. We can merge our batting stats dataset to this one using this key, after some wrangling to make sure the name formats match up.

* **`launch_speed`**

    * This is the speed the ball was traveling the moment it left the bat, measured in MPH. It is also frequently referred to as exit velocity.

* **`launch_angle`**

    * This is the angle the ball was moviing the moment it left the bat, as measured in degrees up from the ground.

* **`hit_distance_sc`**

    * This is the horizontal distance in feet that the home run travels before hitting the some "ground". Since we are only dealing with home runs in Wrigley Field, we know this means it hit the seats. Other stadiums require more or less hit distance for a home run to make it.

* **`pitch_type`**

    * This is the type of pitch that was hit. Information about what each abbrevation means can be found [here](https://library.fangraphs.com/pitch-type-abbreviations-classifications/).

We should now check for inconsistencies or missing values.

In [47]:
hrs.isna().sum()

game_date          0
player_name        0
launch_speed       0
launch_angle       0
hit_distance_sc    0
pitch_type         0
dtype: int64

There are no explicit missing values.

In [48]:
hrs.dtypes

game_date           object
player_name         object
launch_speed       float64
launch_angle         int64
hit_distance_sc      int64
pitch_type          object
dtype: object

The data types of each column are what we would expect, indicating no implicit missing values there.

## Dataset 2 - Batting

To download this dataset in the exact form I am using here, visit [this website](https://www.baseball-reference.com/leagues/majors/2025-standard-batting.shtml), scroll down to the "Player Standard Batting" table, click "Share & Export" and "Get table as CSV (Excel)". I renamed my file to `batting.csv`.

In [41]:
batting = pd.read_csv("data/batting.csv")
batting.head()

,Rk,Player,Age,Team,Lg,WAR,G,PA,AB,R,...,Rbat+,TB,GIDP,HBP,SH,SF,IBB,Pos,Awards,Player-additional
0,1.0,Francisco Lindor#,31.0,NYM,NL,5.9,160.0,732.0,644.0,117.0,...,129.0,300.0,9.0,16.0,0.0,7.0,2.0,*6/DH,ASMVP-10,lindofr01
1,2.0,Rafael Devers*,28.0,2TM,2LG,4.1,163.0,729.0,607.0,99.0,...,139.0,291.0,16.0,6.0,0.0,4.0,10.0,*D3/5H,NaN,deverra01
2,2.0,Rafael Devers*,28.0,BOS,AL,2.3,73.0,334.0,272.0,47.0,...,153.0,137.0,7.0,4.0,0.0,2.0,7.0,D,NaN,deverra01
3,2.0,Rafael Devers*,28.0,SFG,NL,1.9,90.0,395.0,335.0,52.0,...,127.0,154.0,9.0,2.0,0.0,2.0,3.0,D3/5H,NaN,deverra01
4,3.0,Shohei Ohtani*,30.0,LAD,NL,6.6,158.0,727.0,611.0,146.0,...,175.0,380.0,9.0,3.0,0.0,2.0,20.0,*D1,ASMVP-1SS,ohtansh01


Once again, we need to select only the batting stats important to our analysis.

In [42]:
batting.columns

Index(['Rk', 'Player', 'Age', 'Team', 'Lg', 'WAR', 'G', 'PA', 'AB', 'R', 'H',
       '2B', '3B', 'HR', 'RBI', 'SB', 'CS', 'BB', 'SO', 'BA', 'OBP', 'SLG',
       'OPS', 'OPS+', 'rOBA', 'Rbat+', 'TB', 'GIDP', 'HBP', 'SH', 'SF', 'IBB',
       'Pos', 'Awards', 'Player-additional'],
      dtype='object')

In [49]:
batting_cols = ["Player", "WAR", "OBP", "SLG"]
batting = batting[batting_cols]
batting.head()

,Player,WAR,OBP,SLG
0,Francisco Lindor#,5.9,0.346,0.466
1,Rafael Devers*,4.1,0.372,0.479
2,Rafael Devers*,2.3,0.401,0.504
3,Rafael Devers*,1.9,0.347,0.460
4,Shohei Ohtani*,6.6,0.392,0.622


### Data Dictionary

* **`Player`**

    * This is the player whose stats fill out the row. We can merge our home run dataset to this one using this key.

* **`WAR`**

    * Wins Above Replacement; this is essentially how many more wins this player contributes to their team comapared to a "replacement-level" player, i.e. one that could be acquired at little to no cost from another team or the minor leagues.

* **`OBP`**

    * On-Base Percentage measures how often a player is able to get on base.

* **`SLG`**

    * Slugging percentage measures a player's general "slugging" ability, which is how often they are able to hit hard balls. Where something like batting average weighs every hit equally, slugging percentage gives more weight to big hits like home runs. To put it simply, players that have a high SLG are better at hitting home runs, and better at hitting them hard.

These three statistics combined measure each player's general skill with as few variables as possible.

We should also check for explicit missing values.

In [51]:
batting.isna().sum()

Player     0
WAR        1
OBP       92
SLG       92
dtype: int64

There seem to be some missing statistics, most likely for players that didn't play enough for OBP and SLG to be calculated. This is not yet a concern; only players present in both our home run data and batting data need stats. The others will be dropped along the way.

In [52]:
batting.dtypes

Player     object
WAR       float64
OBP       float64
SLG       float64
dtype: object

Every column has the data type we would expect, which is good.

## Dataset 3 - Chicago Weather

To download this dataset in the exact form I am using here, visit [this website](https://www.glerl.noaa.gov/metdata/chi/archive/), scroll down to the "2025" row, right-click "Daily Avg", and select "Save Link As...". I renamed my file to `weather.csv`.

Our final dataset requires some more code to read in. Note that I skip the first 11 rows, as those contain some summary information about the data and would mess up the formatting. If you want to see the full file, you can open it in the `data` folder.

In [53]:
weather = pd.read_csv("data/weather.avg", sep=r"\s+", skiprows=11)
weather.head()

,DOY,WS,WD,AT,n
0,1,11.98,279.0,-0.95,4
1,2,8.61,276.0,-1.33,719
2,3,10.05,295.0,-2.64,719
3,4,9.37,296.0,-5.88,719
4,5,6.81,323.0,-5.54,719


We can drop the `n` column, which represents how many observations were collected on that day (these are daily averages). Every day except the first (where no baseball was played anyway) has 719 observations.

In [54]:
weather = weather.drop(columns=['n'])
weather.head()

,DOY,WS,WD,AT
0,1,11.98,279.0,-0.95
1,2,8.61,276.0,-1.33
2,3,10.05,295.0,-2.64
3,4,9.37,296.0,-5.88
4,5,6.81,323.0,-5.54


### Data Dictionary

* **`DOY`**

    * Day of Year; this represents the day of 2025 the data is for. After some formatting, we can use this column to merge our weather dataset with our home run data.

* **`WS`**

    * Wind speed in meters per second.

* **`WD`**

    * The direction from which the wind is blowing; wind blowing north to south would be zero degrees, south to north would be 180.

* **`AT`**

    * Average temperature in degrees Celsius.

In [55]:
weather.isna().sum()

DOY    0
WS     1
WD     1
AT     1
dtype: int64

From looking at the file myself (it's somewhat small) I know that the missing values here are from day 366. I believe every year has 366 rows to account for leap years, but in the event the year is not a leap year (which is the case for 2025), the 366th row is filled with missing values.

In [56]:
weather.dtypes

DOY      int64
WS     float64
WD     float64
AT     float64
dtype: object

Once again, data types match what we would expect.